In [23]:
!pip install tensorflow --upgrade


In [ ]:
import pandas as pd
dftrain = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/train.csv')
dfeval = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/eval.csv')
import tensorflow as tf

y_train = dftrain.pop('survived')
y_eval = dfeval.pop('survived')
dftrain.head()

,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,male,22.0,1,0,7.2500,Third,unknown,Southampton,n
1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
2,female,26.0,0,0,7.9250,Third,unknown,Southampton,y
3,female,35.0,1,0,53.1000,First,C,Southampton,n
4,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y


In [25]:
CATEGORICAL_COLUMNS = ['sex', 'n_siblings_spouses', 'parch', 'class', 'deck', 'embark_town', 'alone']
NUMERIC_COLUMNS = ['age', 'fare']
feature_columns = []

# Categorical columns
for feature_name in CATEGORICAL_COLUMNS:
    vocabulary = dftrain[feature_name].unique()
    feature_columns.append(
        tf.feature_column.categorical_column_with_vocabulary_list(feature_name, vocabulary)
    )

# Numeric columns
for feature_name in NUMERIC_COLUMNS:
    feature_columns.append(
    tf.feature_column.numeric_column(feature_name, dtype=tf.float32)
    )

print(feature_columns)

[VocabularyListCategoricalColumn(key='sex', vocabulary_list=('male', 'female'), dtype=tf.string, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='n_siblings_spouses', vocabulary_list=(1, 0, 3, 4, 2, 5, 8), dtype=tf.int64, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='parch', vocabulary_list=(0, 1, 2, 5, 3, 4), dtype=tf.int64, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='class', vocabulary_list=('Third', 'First', 'Second'), dtype=tf.string, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='deck', vocabulary_list=('unknown', 'C', 'G', 'A', 'B', 'D', 'F', 'E'), dtype=tf.string, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='embark_town', vocabulary_list=('Southampton', 'Cherbourg', 'Queenstown', 'unknown'), dtype=tf.string, default_value=-1, num_oov_buckets=0), VocabularyListCategoricalColumn(key='alone', vocabulary_list=('n', 'y'), dtype=tf.string, def

# What is epochs ?
- An **epoch** is simply one stream of our entire dataset. The number of epochs we define is the amount of times our model will see the entire of datasets. We use multiple epoches in hope that after seeing the same data multiple times the model will better determine how to estimate it.

# Whats is input function at tensorflow ?
- When you train or evaluate an estimator model (like tf.estimator.LinearClassifier), TensorFlow needs a pipeline to load, batch, shuffle, and repeat data. The input_fn encapsulates this logic and returns a tf.data.Dataset.



In [2]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate, StringLookup, Normalization
from tensorflow.keras.models import Model

# Load data
dftrain = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/train.csv')
dfeval = pd.read_csv('https://storage.googleapis.com/tf-datasets/titanic/eval.csv')
y_train = dftrain.pop('survived')
y_eval = dfeval.pop('survived')

# Cast columns
CATEGORICAL_COLUMNS = ['sex', 'n_siblings_spouses', 'parch', 'class', 'deck', 'embark_town', 'alone']
NUMERIC_COLUMNS = ['age', 'fare']

for col in CATEGORICAL_COLUMNS:
    dftrain[col] = dftrain[col].astype(str)
    dfeval[col] = dfeval[col].astype(str)

for col in NUMERIC_COLUMNS:
    dftrain[col] = dftrain[col].astype('float32')
    dfeval[col] = dfeval[col].astype('float32')

# Preprocessing layers
inputs = {}
encoded_features = []

# ✅ Categorical features with StringLookup
for col in CATEGORICAL_COLUMNS:
    inputs[col] = Input(shape=(1,), name=col, dtype='string')
    vocab = sorted(dftrain[col].unique())  # ensure consistent order
    lookup = StringLookup(vocabulary=vocab, output_mode='one_hot')
    encoded = lookup(inputs[col])
    encoded_features.append(encoded)

# ✅ Numeric features with Normalization
for col in NUMERIC_COLUMNS:
    inputs[col] = Input(shape=(1,), name=col)
    norm = Normalization()
    norm.adapt(dftrain[col].values)
    encoded = norm(inputs[col])
    encoded_features.append(encoded)

# ✅ Combine all encoded features
all_features = Concatenate()(encoded_features)

# Build model
x = Dense(32, activation='relu')(all_features)
x = Dense(16, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)
model = Model(inputs=inputs, outputs=output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# ✅ Input pipeline
def df_to_dataset(data_df, label_df, batch_size=32):
    return tf.data.Dataset.from_tensor_slices((dict(data_df), label_df)).batch(batch_size)

# Train
train_ds = df_to_dataset(dftrain, y_train)
eval_ds = df_to_dataset(dfeval, y_eval)

model.fit(train_ds, epochs=5)
model.evaluate(eval_ds)


Epoch 1/5


ValueError: Exception encountered when calling Functional.call().

[1mInput 0 of layer "dense_3" is incompatible with the layer: expected axis -1 of input shape to have value 41, but received input with shape (None, 1293)[0m

Arguments received by Functional.call():
  • inputs={'sex': 'tf.Tensor(shape=(None,), dtype=string)', 'age': 'tf.Tensor(shape=(None,), dtype=float32)', 'n_siblings_spouses': 'tf.Tensor(shape=(None,), dtype=string)', 'parch': 'tf.Tensor(shape=(None,), dtype=string)', 'fare': 'tf.Tensor(shape=(None,), dtype=float32)', 'class': 'tf.Tensor(shape=(None,), dtype=string)', 'deck': 'tf.Tensor(shape=(None,), dtype=string)', 'embark_town': 'tf.Tensor(shape=(None,), dtype=string)', 'alone': 'tf.Tensor(shape=(None,), dtype=string)'}
  • training=True
  • mask={'sex': 'None', 'age': 'None', 'n_siblings_spouses': 'None', 'parch': 'None', 'fare': 'None', 'class': 'None', 'deck': 'None', 'embark_town': 'None', 'alone': 'None'}
  • kwargs=<class 'inspect._empty'>